In [2]:
# base
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic() 
model = "claude-haiku-4-5"

In [3]:
# Helpers
def add_user_message(messages, text):
  user_message = {
    "role": "user", 
    "content": text
  }
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {
    "role": "assistant",
    "content": text
  }
  messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
  params= {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
    "stop_sequences": stop_sequences
  }

  if system:
    params["system"] = system

  message = client.messages.create(**params)
  return message.content[0].text

In [ ]:
# generate dataset
import json

def generate_dataset():
  prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [5]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
  messages = []
  add_user_message(messages, prompt)
  output = chat(messages)
  return output

In [6]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the result"""
  output = run_prompt(test_case)
  
  # TODO - Grading
  score = 10

  return {
    "output": output,
    "test_case": test_case,
    "score": score
  }



In [7]:
def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  return results

In [10]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses an S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse an AWS S3 bucket name from an S3 URI.\n    \n    Args:\n        s3_uri: S3 URI in the format 's3://bucket-name/key/path'\n    \n    Returns:\n        The S3 bucket name\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \n    Example:\n        >>> parse_s3_bucket_name('s3://my-bucket/path/to/object')\n        'my-bucket'\n    \"\"\"\n    # Remove leading/trailing whitespace\n    s3_uri = s3_uri.strip()\n    \n    # Validate format\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(f\"Invalid S3 URI format: {s3_uri}. Must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Extract bucket name (everything before the first '/')\n    bucket_name = uri_without_prefix.split('/')[0]\n    \n    # Val